# GTSEP stochastic
GTSEP stochastic. same as GSTEP v1a multi, but this is long-term stochastic introducing scenarios $\Omega_y$, a set of scenarios for each representative year $y$.

### Indexes and index sets

- $n \in N$: Set of nodes.
- $i \in G^{old}$: Set of existing generators (at node $n$).
- $i \in G^{new}$: Set of new generators (at node $n$).
- $i \in G$: Set of all generators.
- $i \in G_n$: Set of generators at node $n$, including new generators.
- $b \in B^{new}$: Set of new branches.
- $b \in B^{old}$: Set of existing branches.
- $b \in B$: Set of all branches.
- $b \in B_n^{in}$: Set of branches coming into node $n$, including new branches.
- $b \in B_n^{out}$: Set of branches going out of node $n$, including new branches.
- $s \in S^{old}$: Set of old batteries (at node $n$).
- $s \in S^{new}$: Set of new batteries (at node $n$).
- $s \in S$: Set of all batteries (at node $n$).
- $s \in S_n$: Set of batteries at node $n$, including new batteries.
- $\omega \in \Omega_y$: Set of scenarios in year y.
- $y \in Y$: Set of years.
- $y' \in Y_y$: Set of years up to year $y$. $Y_y = \{y' \in Y \mid y' \leq y\}$.
- $w \in W$: Set of representative weeks.
- $t \in T_w$: Set of hourly periods within each representative week $w$.

### Parameters

- $P_{i}^{\min}$: Min power output generator $i$ (MW)
- $P_{i}^{\max}$: Max power output generator $i$ (MW)
- $VOLL$: Value of lost load (\$/MWh)
- $CC$: Cost of curtailment (\$/MWh)
- $MC_{i,y}$: Marginal cost of generator $i$ (\$/MWh)
- $CO2_{i,y}$: Cost of CO2 emissions generator $i$ (\$/MWh)
- $E_{i,y}$: CO2 emissions generator $i$ (ton/MWh)
- $E_{limit}$: CO2 emissions limit (ton)
- $D_{n,\omega,y,w,t}$: Demand at node $n$, scenario $\omega$, year $y$, week $w$, hour $t$ (MW)
- $l_b$: Loss factor of branch $b$
- $P_{b,y}^{\max}$: Max power flow branch $b$ in year $y$ (MW)
- $\eta_{s}^{ch}$: Charge efficiency battery $s$
- $\eta_{s}^{dis}$: Discharge efficiency battery $s$
- $P_{s}^{ch,\max}, P_{s}^{ch,\min}$: Charging limits battery $s$ (MW)
- $P_{s}^{dis,\max}, P_{s}^{dis,\min}$: Discharging limits battery $s$ (MW)
- $SOC_{s}^{\max}, SOC_{s}^{\min}$: SOC limits battery $s$ (MWh)
- $MC_{s,y}^{dis}$: Marginal discharge cost battery $s$ (\$/MWh)
- $cf_{i,y,w,t}$: Capacity factor of generator $i$ at year $y$, week $w$, hour $t$
- $P_b^{min}, P_b^{\max}$: Min/max capacity of new branch $b$ (MW)
- $AIC_{i,y}, AIC_{s,y}, AIC_{b,y}$: Annualized investment costs generators, batteries, branches (\$/MW or \$/MWh)
- $weight_w$: Weight of representative week $w$
- $p_{\omega,y}$: Probability of scenario $\omega$ in year $y$.

### Decision variables

- $g_{i,\omega,y,w,t}$: Generation dispatch generator $i$ (MW)
- $f_{b,\omega,y,w,t}$: Power flow branch $b$ (MW)
- $sh_{n,\omega,y,w,t}$: Load shedding at node $n$ (MW)
- $c_{i,\omega,y,w,t}$: Curtailment generator $i$ (MW)
- $g_{s,\omega,y,w,t}^{ch}, g_{s,\omega,y,w,t}^{dis}$: Charge/discharge battery $s$ (MW)
- $soc_{s,\omega,y,w,t}$: State of charge battery $s$ (MWh)
- $soc_{s,y}^{max}$: Built storage capacity battery $s$ (MWh)
- $p_{i,y}^{max}$: Built capacity new generator $i$ (MW)
- $p_{b,y}^{max}$: Built capacity new branch $b$ (MW)

## Optimization Model

## Objective function

**Minimize:**
$$
\sum_{y \in Y} \left( AIC_y + \sum_{\omega \in \Omega} p_{\omega,y} OC_{\omega,y} \right)
$$

where

$$
OC_{\omega,y} = \sum_{w \in W}weight_w\left[
\sum_{i\in G}\sum_{t\in T_w}(MC_{i,y}+CO2_{i,y})g_{i,\omega,y,w,t} +
\sum_{s\in S}\sum_{t\in T_w}MC_{s,y}^{dis}g_{s,\omega,y,w,t}^{dis}\eta_s^{dis} +
\sum_{n\in N}\sum_{t\in T_w}sh_{n,\omega,y,w,t}VOLL +
\sum_{i\in G}\sum_{t\in T_w}c_{i,\omega,y,w,t}CC
\right]
$$

and

$$
AIC_y = \sum_{i\in G^{new}}AIC_{i,y}p_{i,y}^{max} + 
\sum_{b\in B^{new}}AIC_{b,y}p_{b,y}^{max} +
\sum_{s\in S^{new}}AIC_{s,y}soc_{s,y}^{max}
$$

---

### Constraints

1. **Power balance**

$$
\sum_{i \in G_n}(g_{i,\omega,y,w,t} - c_{i,\omega,y,w,t}) + \sum_{b \in B_n^{in}} f_{b,\omega,y,w,t}(1 - l_{b}) - \sum_{b \in B_n^{out}} f_{b,\omega,y,w,t} - \sum_{s \in S_n}(g_{s,\omega,y,w,t}^{ch} - \eta_{s}^{dis}g_{s,\omega,y,w,t}^{dis}) + sh_{n,\omega,y,w,t} = D_{n,\omega,y,w,t}, \quad \forall n,\omega,y,w,t
$$

2. **Load shedding limit**

$$
sh_{n,\omega,y,w,t} \leq D_{n,\omega,y,w,t}, \quad \forall n,\omega,y,w,t
$$

3. **Curtailment limit**

$$
0 \leq c_{i,\omega,y,w,t} \leq g_{i,\omega,y,w,t}, \quad \forall i,\omega,y,w,t
$$

4. **Generator output and installed capacity limits**

a. Existing generators:

$$
P_{i}^{\min} \leq g_{i,\omega,y,w,t} \leq P_{i}^{\max} cf_{i,y,w,t}, \quad \forall i \in G^{old},\omega,y,w,t
$$

b. New generators dispatch:

$$
0 \leq g_{i,\omega,y,w,t} \leq cf_{i,y,w,t}\sum_{y' \in Y_y} p_{i,y'}^{max}, \quad \forall i \in G^{new},\omega,y,w,t
$$

c. New generators installed capacity:

$$
p_{i,y}^{max} \leq P_{i}^{\max}, \quad \forall i \in G^{new}, y
$$

5. **Branch flow and installed capacity limits**

a. Existing branches:

$$
-P_{b}^{\max} \leq f_{b,\omega,y,w,t} \leq P_{b}^{\max}, \quad \forall b \in B^{old},\omega,y,w,t
$$

b. New branches dispatch:

$$
-\sum_{y' \in Y_y} p_{b,y'}^{max} \leq f_{b,\omega,y,w,t} \leq \sum_{y' \in Y_y} p_{b,y'}^{max}, \quad \forall b \in B^{new},\omega,y,w,t
$$

c. New branches installed capacity:

$$
p_{b,y}^{max} \leq P_{b}^{\max}, \quad \forall b \in B^{new}, y
$$

6. **Emissions restriction**

$$
\sum_{w \in W} \sum_{t \in T_w} \sum_{i \in G} E_{i,y} g_{i,\omega,y,w,t} \leq E_{limit}, \quad \forall \omega \in \Omega, y \in Y
$$

7. **Battery operational constraints**

a. Charging limits:

$$
P_{s}^{ch,\min} \leq g_{s,\omega,y,w,t}^{ch} \leq P_{s}^{ch,\max}, \quad \forall s \in S^{old},\omega,y,w,t
$$

$$
0 \leq g_{s,\omega,y,w,t}^{ch} \leq \frac{\sum_{y' \in Y_y} soc_{s,y'}^{max}}{batt_{hours} \cdot cdrate}, \quad \forall s \in S^{new},\omega,y,w,t
$$

b. Discharging limits:

$$
P_{s}^{dis,\min} \leq g_{s,\omega,y,w,t}^{dis} \leq P_{s}^{dis,\max}, \quad \forall s \in S^{old},\omega,y,w,t
$$

$$
0 \leq g_{s,\omega,y,w,t}^{dis} \leq \frac{\sum_{y' \in Y_y} soc_{s,y'}^{max}}{batt_{hours}}, \quad \forall s \in S^{new},\omega,y,w,t
$$

c. State of charge limits:

$$
SOC_{s}^{\min} soc_{s,y}^{max} \leq soc_{s,\omega,y,w,t} \leq SOC_{s}^{\max} soc_{s,y}^{max}, \quad \forall s,\omega,y,w,t
$$

d. State of charge dynamics:

$$
soc_{s,\omega,y,w,t} = soc_{s,\omega,y,w,t-1} + \eta_{s}^{ch} g_{s,\omega,y,w,t}^{ch} - \frac{1}{\eta_{s}^{dis}} g_{s,\omega,y,w,t}^{dis}, \quad \forall s,\omega,y,w,t \in T_w \setminus \{0\}
$$

e. Initial state of charge:

$$
soc_{s,\omega,y,w,0} = SOC_{s}^{\min} soc_{s,y}^{max}, \quad \forall s,\omega,y,w
$$

f. End of week state of charge:

$$
soc_{s,\omega,y,w,T_w[-1]} = SOC_{s}^{\min} soc_{s,y}^{max}, \quad \forall s,\omega,y,w
$$

8. **Variable definitions**

All continuous variables are non-negative.

$$
g_{i,\omega,y,w,t}, f_{b,\omega,y,w,t}, sh_{n,\omega,y,w,t}, c_{i,\omega,y,w,t}, g_{s,\omega,y,w,t}^{ch}, g_{s,\omega,y,w,t}^{dis}, soc_{s,\omega,y,w,t}, p_{i,y}^{max}, p_{b,y}^{max}, soc_{s,y}^{max} \geq 0
$$


